# Solutions · 04 · Yield-stress fluids

Worked solutions to the exercises of [notebook 04](../notebooks/04_yield_stress_fluids.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engrheo is missing): install it from GitHub
    import engrheo
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engrheo"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engrheo import datasets, fitting, geometry, models
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

Drilling engineers estimate the true yield stress with the low-shear yield point LSYP = $2\theta_3 - \theta_6$ (lbf/100 ft², × 0.4788 for Pa). Compute it for the mud and compare with the API yield point, the Herschel-Bulkley $\tau_y$ and the true value (4.8 Pa).

In [2]:
mud = pd.read_csv(datasets.path("drilling_mud_fann.csv"), comment="#")
th = dict(zip(mud.rpm, mud.dial_reading_deg))
lsyp = (2 * th[3] - th[6]) * 0.4788
api = geometry.api_bingham(th[600], th[300])
tau_m, rate_m = geometry.fann35(mud.rpm.to_numpy(), mud.dial_reading_deg.to_numpy())
o = np.argsort(rate_m)
hb = fitting.fit_flow_curve(rate_m[o], tau_m[o], "herschel_bulkley")
for name, v in (("API yield point", api["tau_y"]), ("low-shear yield point 2 theta3 - theta6", lsyp),
                ("Herschel-Bulkley fit", hb.params["tau_y"]), ("true yield stress", 4.8)):
    print(f"{name:<38}: {v:5.2f} Pa")

API yield point                       : 12.21 Pa
low-shear yield point 2 theta3 - theta6:  5.27 Pa
Herschel-Bulkley fit                  :  5.15 Pa
true yield stress                     :  4.80 Pa


The low-shear yield point, built from the two lowest readings, lands close to the Herschel-Bulkley yield
stress and the true value; the API yield point, extrapolated from the two highest readings, is more than
twice too large. That is why drilling practice increasingly reports LSYP or Herschel-Bulkley parameters
for hole cleaning and suspension calculations.

## Exercise 2

Fit the Herschel-Bulkley model to the chocolate data using only the rates above 10 1/s. How much does the yield stress change? What does that say about extrapolated yield stresses?

In [3]:
choc = pd.read_csv(datasets.path("chocolate_casson.csv"), comment="#")
rc, tc = choc.shear_rate_1_s.to_numpy(), choc.shear_stress_Pa.to_numpy()
for lo in (0.0, 10.0):
    m = rc >= lo
    f = fitting.fit_flow_curve(rc[m], tc[m], "herschel_bulkley")
    print(f"HB fitted to rates >= {lo:4.0f} 1/s ({m.sum()} points): tau_y = {f.params['tau_y']:5.2f} +/- {f.se['tau_y']:.2f} Pa")

HB fitted to rates >=    0 1/s (10 points): tau_y = 17.74 +/- 1.30 Pa
HB fitted to rates >=   10 1/s (5 points): tau_y = 23.56 +/- 9.93 Pa


Dropping the low-rate points changes the extrapolated yield stress and makes it much more uncertain: the
further the data are from zero shear rate, the longer the extrapolation. An extrapolated yield stress is
only as good as the lowest shear rates measured.

## Exercise 3

**Implement it yourself:** fit the Bingham model to the concrete data by an ordinary straight-line fit of stress against rate (`np.polyfit`) and compare with the library's log-residual fit. Which points does each weight most?

In [4]:
conc = pd.read_csv(datasets.path("concrete_flow_curve.csv"), comment="#")
mu_p, ty = np.polyfit(conc.shear_rate_1_s, conc.shear_stress_Pa, 1)
lib = fitting.fit_flow_curve(conc.shear_rate_1_s, conc.shear_stress_Pa, "bingham")
print(f"straight line (absolute residuals): tau_y = {ty:.1f} Pa, mu_p = {mu_p:.2f} Pa s")
print(f"library (relative residuals):       tau_y = {lib.params['tau_y']:.1f} Pa, mu_p = {lib.params['mu_p']:.2f} Pa s")
print("true: tau_y = 600 Pa, mu_p = 45 Pa s")

straight line (absolute residuals): tau_y = 631.0 Pa, mu_p = 42.16 Pa s
library (relative residuals):       tau_y = 623.9 Pa, mu_p = 43.10 Pa s
true: tau_y = 600 Pa, mu_p = 45 Pa s


Ordinary least squares weights absolute errors, so the high-rate points (largest stresses) dominate the
slope; relative residuals weight every point equally in percentage terms. Here the stresses vary only by a
factor of about 2, so the two answers are close; for data spanning decades the choice matters much more
(notebook 03).